# Appendix 1: Code example

This section includes examples for how to calculate the persistent diagram based features (the first 3 bits of the PHF) using giotta-tda [[cite]] and examples for how to do this using the G\&T package in python.

Example code to calculate the persistence diagram of benzene. First we load our packages (note that using graphs-and-topology-for-chemists.py will do this for you.

In [1]:
from rdkit import Chem
import rdkit.Chem.AllChem as AllChem
from gtda.homology import VietorisRipsPersistence
from gtda.diagrams import PersistenceEntropy
import numpy as np

This function creates coordinates from  `rdkit`'s molecular structure. It is part of `G&T`, but shown here in full for educational purposes.

In [2]:
def generate_structure_from_smiles(smiles):

    # Generate a 3-D structure from smiles

    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)

    status = AllChem.EmbedMolecule(mol)
    status = AllChem.UFFOptimizeMolecule(mol)

    conformer = mol.GetConformer()
    coordinates = conformer.GetPositions()
    coordinates = np.array(coordinates)

    return coordinates 

This function creates the Vietoris-Rips complex, calculates the persistence over length scales and creates the persistence diagram.

In [5]:
def smiles_to_persistence_diagrams(smiles):
    coords=generate_structure_from_smiles(smiles)
    # makes a point cloud version of the structure
    # there are no atom types

    # Track connected components, loops, and voids
    homology_dimensions = [0, 1, 2]

    # Peristence calculation
    persistence = VietorisRipsPersistence(
        metric="euclidean",
        homology_dimensions=homology_dimensions,
        n_jobs=6,
        collapse_edges=True,
    )
    reshaped_coords=coords[None, :, :]
    diagrams_basic=persistence.fit_transform(reshaped_coords)
    return coords, diagrams_basic


# Creating the topological features

This code block is all you need to type to go from a SMILES string to the small data input (just persistence entropy).

In [7]:
smiles="c1ccccc1"
coords, diagrams_basic=smiles_to_persistence_diagrams(smiles)
persistence_entropy = PersistenceEntropy()
# calculate topological feature matrix
X_basic = persistence_entropy.fit_transform(diagrams_basic)



see `Analysis of persistence homology for molecules.ipynb` for more examples